In [1]:

import csv
import pymysql
from datetime import datetime

In [3]:
# Database connection
conn = pymysql.connect(
    host="mysql",
    port=3306,
    user="root",
    password="OSUGoPokes",
    database="IEM4723 Project"
)

cursor = conn.cursor()

In [5]:

csv_path = r"C:\Users\kailu\Downloads\BostonCrime1Yr.csv"
BATCH_SIZE = 1000

total_inserted = 0
batch_number = 0

try:
    with open(csv_path, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)

        # Use first row as column names
        headers = next(reader)
        headers = [h.strip().strip('"') for h in headers]

        print("CSV columns:", headers)

        # Create columns based on the CSV headers.
        # All columns are initially VARCHAR to preserve source values.
        column_definitions = ", ".join(
            f"`{header.replace('`', '``')}` TEXT"
            for header in headers
        )

        create_query = f"""
        CREATE TABLE BostonCrime1Yr (
            {column_definitions}
        )
        """
        cursor.execute(create_query)
        conn.commit()

        print("Table BostonCrime1Yr created successfully.")

except Exception as e:
    print(f"Table Creation failed: {e}")


Table Creation failed: [Errno 2] No such file or directory: 'C:\\Users\\kailu\\Downloads\\BostonCrime1Yr.csv'


In [ ]:
try:
        #INSERT statement using the CSV headers
        quoted_columns = ", ".join(
            f"`{h.replace('`', '``')}`" for h in headers
        )
        placeholders = ", ".join(["%s"] * len(headers))

        insert_query = f"""
        INSERT INTO BostonCrime1Yr ({quoted_columns})
        VALUES ({placeholders})
        """

        batch = []

        # The header row is already consumed, so data starts at row 2
        for row in reader:
            if not row:
                continue

            if len(row) != len(headers):
                print(
                    f"Skipping malformed row: expected {len(headers)} "
                    f"columns, received {len(row)}."
                )
                continue

            # Convert empty fields to SQL NULL
            batch.append(tuple(value if value != "" else None
                               for value in row))

            if len(batch) >= BATCH_SIZE:
                batch_number += 1

                try:
                    cursor.executemany(insert_query, batch)
                    conn.commit()
                    total_inserted += len(batch)

                    print(
                        f"Batch {batch_number}: SUCCESS - "
                        f"{len(batch)} rows inserted. "
                        f"Total: {total_inserted}"
                    )

                except Exception as e:
                    conn.rollback()
                    print(f"Batch {batch_number}: ERROR - {e}")
                    raise

                batch = []

        # Insert the final partial batch
        if batch:
            batch_number += 1

            try:
                cursor.executemany(insert_query, batch)
                conn.commit()
                total_inserted += len(batch)

                print(
                    f"Batch {batch_number}: SUCCESS - "
                    f"{len(batch)} rows inserted. "
                    f"Total: {total_inserted}"
                )

            except Exception as e:
                conn.rollback()
                print(f"Batch {batch_number}: ERROR - {e}")
                raise

        print(
            f"\nImport complete. Total rows inserted: "
            f"{total_inserted}"
        )

except Exception as e:
    print(f"Import failed: {e}")

finally:
    if "cursor" in locals():
        cursor.close()
    conn.close()

